# SST Indices Preprocessing

This notebook runs the generalized SST index preprocessing workflow. It calls `scripts/run_process_sst_index.py` to:
1. Load raw monthly data for E3SM, observations (HadISST2), and CESM-SMYLE.
2. Compute monthly and seasonal averages for 13 SST indices (including Nino regions, TNA, TSA, IOD, TNI, ONI, RONI, and Atlantic indices).
3. Save the results as NetCDF files to the diagnostic output directories.

This allows the downstream diagnostics notebook `4_refactor_sst_skill_ts.ipynb` to load these precalculated indices instantly.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Identify repository root
REPO_ROOT = Path(os.getcwd())
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = Path("..")

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
# Multi-case E3SM hindcasts. Add more entries here as new post-processed
# hindcasts land under data_dir with the same directory convention.
E3SM_CASES = {
    "E3SM-FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "cache_tag": "JRA55_FOSIRL",
        "display_name": "E3SMv3-FOSIRL",
    },
    "E3SM-BruteForce": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "cache_tag": "BruteForce",
        "display_name": "E3SMv3-Reanalysis",
    },
#    "E3SM-4DEnVarOcn": {
#        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
#        "cache_tag": "4DEnVarOcn",
#        "display_name": "E3SMv3-4DEnVarOcn",
#    },
}

CONFIG = {
    "sources": ["obs", "e3sm", "smyle"],
    "e3sm_cases": E3SM_CASES,
    "e3sm_data_dir": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
    "regions": [
        "IOD", "TNI", "ONI", "RONI",
        "Nino12", "Nino3", "Nino3.4", "Nino4",
        "TNA", "TSA", "PACWRAMPOOL", "AtlNino", "AtlMDR"
    ],  # List of target indices/regions to compute
    "custom_regions": {
        "Nino12": {
            "lonlat": [270.0, 280.0, -10.0, 0.0],
            "long_name": "Nino 1+2 regional mean SST",
        },
        "Nino3": {
            "lonlat": [210.0, 270.0, -5.0, 5.0],
            "long_name": "Nino 3 regional mean SST",
        },
        "Nino3.4": {
            "lonlat": [190.0, 240.0, -5.0, 5.0],
            "long_name": "Nino 3.4 regional mean SST",
        },
        "Nino4": {
            "lonlat": [160.0, 210.0, -5.0, 5.0],
            "long_name": "Nino 4 regional mean SST",
        },
        "TNA": {
            "lonlat": [305.0, 345.0, 5.0, 25.0],
            "long_name": "TNA regional mean SST",
        },
        "TSA": {
            "lonlat": [330.0, 10.0, -20.0, 0.0],
            "long_name": "TSA regional mean SST",
        },
        "PACWRAMPOOL": {
            "lonlat": [60.0, 170.0, -15.0, 15.0],
            "long_name": "PACWRAMPOOL regional mean SST",
        },
        "AtlNino": {
            "lonlat": [340.0, 360.0, -3.0, 3.0],
            "long_name": "Atlantic Nino regional mean SST",
        },
        "AtlMDR": {
            "lonlat": [280.0, 350.0, 10.0, 20.0],
            "long_name": "Atlantic MDR regional mean SST",
        },
        # Helper regions for derived indices
        "IOD_West": {
            "lonlat": [50.0, 70.0, -10.0, 10.0],
            "long_name": "IOD West regional mean SST",
        },
        "IOD_East": {
            "lonlat": [90.0, 110.0, -10.0, 0.0],
            "long_name": "IOD East regional mean SST",
        },
        "TropicalMean": {
            "lonlat": [0.0, 360.0, -20.0, 20.0],
            "long_name": "Tropical Mean regional mean SST",
        },
    },
    "outdir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/E3SMLE",
    "smyle_outdir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE",
    "init_months": [5, 11],
    "year_start": 1980,
    "year_end": 2018,
    "climy0": 1980,
    "climy1": 2010,
    "nlead": 24,
    "e3sm_nens": 10,
    "smyle_nens": 20,
    "workers": 8,
    "force": True,  # Set to True to force rewrite
}


In [ ]:
# -----------------------------
# Construct and execute command for each index/case
# -----------------------------
import json

script_path = str(REPO_ROOT / "scripts" / "run_process_sst_index.py")

# Set environment variables for GDAL/PROJ
env = os.environ.copy()
conda_prefix = "/global/homes/z/zhan391/.conda/envs/e3sm_analysis"
env["GDAL_DATA"] = f"{conda_prefix}/share/gdal"
env["PROJ_LIB"] = f"{conda_prefix}/share/proj"


def build_base_cmd(sources, region):
    cmd = [
        sys.executable,
        script_path,
        "--sources", *sources,
        "--outdir", CONFIG["outdir"],
        "--smyle-outdir", CONFIG["smyle_outdir"],
        "--init-months", *(str(m) for m in CONFIG["init_months"]),
        "--year-start", str(CONFIG["year_start"]),
        "--year-end", str(CONFIG["year_end"]),
        "--climy0", str(CONFIG["climy0"]),
        "--climy1", str(CONFIG["climy1"]),
        "--nlead", str(CONFIG["nlead"]),
        "--e3sm-nens", str(CONFIG["e3sm_nens"]),
        "--smyle-nens", str(CONFIG["smyle_nens"]),
        "--workers", str(CONFIG["workers"]),
        "--regions", region,
    ]

    if CONFIG.get("custom_regions"):
        cmd.extend(["--custom-regions", json.dumps(CONFIG["custom_regions"])])

    if CONFIG["force"]:
        cmd.append("--force")

    return cmd


def run_cmd(cmd, label, region):
    print("=" * 60)
    print(f"Processing SST index: {region} | {label}")
    print("=" * 60)
    print("Running command:")
    print(" ".join(cmd))

    result = subprocess.run(cmd, env=env, capture_output=True, text=True)

    print("\n--- STDOUT ---")
    print(result.stdout)

    if result.returncode != 0:
        print("\n--- STDERR ---")
        print(result.stderr)
        raise RuntimeError(
            f"Preprocessing failed for '{label}' index '{region}' "
            f"with exit code {result.returncode}"
        )


shared_sources = [s for s in CONFIG["sources"] if s != "e3sm"]
process_e3sm = "e3sm" in CONFIG["sources"]

for r in CONFIG["regions"]:
    if shared_sources:
        cmd = build_base_cmd(shared_sources, r)
        run_cmd(cmd, "+".join(shared_sources), r)

    if process_e3sm:
        for case_key, case_info in CONFIG["e3sm_cases"].items():
            cmd = build_base_cmd(["e3sm"], r)
            cmd.extend([
                "--e3sm-data-dir", CONFIG["e3sm_data_dir"],
                "--e3sm-case-prefix", case_info["case_prefix"],
                "--e3sm-cache-tag", case_info["cache_tag"],
                "--e3sm-display-name", case_info.get("display_name", case_key),
            ])
            run_cmd(cmd, case_key, r)

print("\nAll SST indices processed successfully!")
